# ROAD-SHIELD — fine-tune the pothole/crack detector (real, GPU)

This notebook does the **real** detector training that cannot be done in a CPU sandbox, and runs this
repository's own **verification gate** so the result is honest end-to-end:

1. train YOLOv8s on human-labelled pothole/crack data (duplicate-clustered splits, measured-only metrics)
2. export `best.onnx`
3. run the **artefact check** (does the served ONNX reproduce the trained network within 0.02 mAP?)
4. download `best.onnx` + `detector_metrics.json` + `artefact_check.json`
5. install them into the repo, run the clean-road check, commit — boxes flip **candidate → verified** on their own

**Runtime: GPU.** Runtime ▸ Change runtime type ▸ **T4 GPU**. Training ~1–2 h for 100 epochs.
Nothing here fabricates a number: every metric comes from Ultralytics on a held-out split, and the gate
either passes on its merits or it doesn't.


## 1. Setup — install + clone the repo (master has the scripts)


In [ ]:
!nvidia-smi -L    # confirm a GPU is attached
!pip -q install ultralytics opencv-python-headless pyyaml kagglehub
!git clone -q --depth 1 https://github.com/udbhav968-creator/SIH_PROJECT.git /content/repo
%cd /content/repo
!ls training scripts colab | head


## 2. Get human-labelled detection data

You need **bounding-box labels made by people** (not classification-only sets). Two options — use either or both:

**(a) Kaggle pothole/crack sets** — upload your `kaggle.json` (Kaggle ▸ Settings ▸ API ▸ Create token) to the
Colab Files panel, then run the Kaggle cell. Replace the example datasets with ones you have checked the licence of.

**(b) RDD2022-India** — if you have the RDD2022 archive, point the fetch script at it. RDD2022 is distributed
from figshare/CRDDC; download it yourself and pass the local zip.


In [ ]:
# (a) Kaggle labelled sets  -- one --dataset per source: "<owner>/<slug>:<target class>"
import os
if os.path.exists('/content/kaggle.json'):
    !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !python training/prepare_kaggle_detection_data.py --out /content/data \
        --dataset andrewmvd/pothole-detection:pothole \
        --dataset chitholian/annotated-potholes-dataset:pothole
else:
    print('No kaggle.json uploaded — skip this cell if you are using RDD2022 below.')


In [ ]:
# (b) OPTIONAL RDD2022-India  -- only if you have the archive (skip if using Kaggle only)
# Upload the RDD2022 zip to Colab, then set the path:
RDD_ZIP = ''   # e.g. '/content/RDD2022.zip'  (leave '' to skip)
if RDD_ZIP:
    !python scripts/fetch_rdd2022_countries.py India --local-zip "$RDD_ZIP"
    print('RDD2022-India prepared. Merge its YOLO folder into /content/data or train on it directly.')


## 3. Record each source's licence (honesty: no unlicensed data ships)


In [ ]:
import json, os
p = '/content/data/sources.json'
if os.path.exists(p):
    s = json.load(open(p))
    for k, v in s.items(): print(k, '|', v.get('url'), '|', v.get('licence'), '|', v.get('n_images'))
    LICENCES = {  # fill in the licence text from each dataset's page
        # '<source name printed above>': '<licence>',
    }
    for k, lic in LICENCES.items(): s[k]['licence'] = lic
    json.dump(s, open(p, 'w'), indent=2)
    print('\nlicences updated')
else:
    print('no sources.json yet — run the data cell first')


## 4. Train (GPU) — measured-only, duplicate-clustered splits
`best.pt`, `best.onnx`, `detector_metrics.json`, `data.yaml` and the splits are written to `/content/out`.


In [ ]:
!python training/train_detectors_real.py --data-root /content/data --out /content/out --epochs 100 --imgsz 640


In [ ]:
import json; m = json.load(open('/content/out/detector_metrics.json'))
print('TEST  mAP@0.5 :', m['test']['map50'])
print('TEST  mAP@0.5-0.95 :', m['test']['map50_95'])
print('per class:', json.dumps(m['test']['per_class'], indent=2))
print('train/val/test images:', m['split']['train']['images'], m['split']['val']['images'], m['split']['test']['images'])


## 5. Verification gate (in Colab) — does the ONNX reproduce the trained net?
This is the exact check (`scripts/verify_rdd_detector.py`) that keeps unverified boxes hidden. It scores the
exported `best.onnx` through the repo's own decoder on the held-out test split and compares to `best.pt`.
Writes `/content/result/artefact_check.json` with `passed: true/false`.


In [ ]:
!python colab/colab_verify_detector.py
import json, os
ac = '/content/result/artefact_check.json'
if os.path.exists(ac):
    a = json.load(open(ac))
    print('PASSED :', a.get('passed'))
    print('onnx test mAP@0.5 :', a.get('onnx_test_map50'), ' vs best.pt :', a.get('best_pt_test_map50'))
    print('serving decoder vs best.pt :', a.get('serving_decoder_vs_best_pt'))


## 6. Download the three artefacts
Do **not** edit them — the installer checks the ONNX's SHA-256 against the measured metrics.


In [ ]:
from google.colab import files
for f in ['/content/out/detector_metrics.json',
          '/content/out/train/weights/best.onnx',
          '/content/result/artefact_check.json']:
    try: files.download(f)
    except Exception as e: print('download', f, 'failed:', e)


## 7. Install into the repo on your PC → boxes become **verified**
On your machine, in the repo, put the three files in a folder (e.g. `C:\\Users\\Dell\\Downloads`), then:

```powershell
cd C:\Users\Dell\SIH_PROJECT
python -m scripts.install_trained_detector `
    --metrics  "$HOME\Downloads\detector_metrics.json" `
    --onnx     "$HOME\Downloads\best.onnx" `
    --artefact "$HOME\Downloads\artefact_check.json"

# place best.pt + the test split where the clean-road check expects them, then:
python -m scripts.verify_rdd_detector --clean-roads

git add -A
git commit -m "Install fine-tuned RDD detector (measured) + verification"
git push origin master
```

Once `artefact_check.json` shows `passed: true` and the clean-road check passes, the pipeline serves the
road-damage boxes as **verified** automatically — no code change. If the artefact check fails by a hair, the
most common cause is the two mAP numbers coming from different scorers; `docs/DETECTION_PIPELINE.md` explains
how to re-export with a pinned opset/size and re-score both the same way before trusting the gap.
